In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix)
from scipy.stats import randint

In [5]:
# Load Dataset

df = pd.read_csv("bank_marketing.csv")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,32,admin.,married,secondary,no,11434,no,no,cellular,14,jan,152,3,259,10,unknown,no
1,87,student,single,secondary,no,-1149,no,yes,cellular,9,mar,911,11,195,2,failure,no
2,62,housemaid,single,primary,no,14361,no,no,cellular,27,nov,2563,12,-1,0,unknown,yes
3,23,unemployed,single,secondary,no,28512,no,no,cellular,27,jun,696,12,-1,0,unknown,yes
4,27,housemaid,married,secondary,no,58589,yes,no,cellular,18,apr,2834,11,-1,0,unknown,yes


In [6]:
print(df.shape)

# Print column name

print(df.columns.tolist)

(20000, 17)
<bound method IndexOpsMixin.tolist of Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')>


In [7]:
df.columns = df.columns.str.strip()
print(df.columns)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')


In [8]:
target = "y"

# Check missing values

print(df.isnull().sum())

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


In [11]:
# Check Data Types

df.dtypes

age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

In [9]:
print(df[target].value_counts())

y
yes    13307
no      6693
Name: count, dtype: int64


In [12]:
# Label Encoding

encoder = LabelEncoder()

for col in df.select_dtypes(include="str").columns:
    df[col] = encoder.fit_transform(df[col])

In [14]:
# Feature & Target

X = df.drop(columns=[target])
y = df[target]
print("Feature Shape = ",X.shape)
print("Target Shape = ",y.shape)

Feature Shape =  (20000, 16)
Target Shape =  (20000,)


In [17]:
# Train-Test Split (70% Train, 30% Test)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [19]:
print("Traning :", X_train.shape)
print("Testing  :", X_test.shape)

Traning : (16000, 16)
Testing  : (4000, 16)


In [22]:
# Decision Tree

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Make Prediction

Prediction = model.predict(X_test)

# Check Accuracy

print("Accuracy :", accuracy_score(y_test, Prediction))

# Classification Report

print(classification_report(y_test, Prediction))

# Confusion Matrix

print("confusion_matrix :- \n",confusion_matrix(y_test, Prediction))

Accuracy : 0.73075
              precision    recall  f1-score   support

           0       0.58      0.64      0.61      1308
           1       0.82      0.77      0.79      2692

    accuracy                           0.73      4000
   macro avg       0.70      0.71      0.70      4000
weighted avg       0.74      0.73      0.73      4000

confusion_matrix :- 
 [[ 841  467]
 [ 610 2082]]


In [23]:
# Cross Validation (Stratified K-Fold)

cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [28]:
# Cross Validation Score

scores = cross_val_score(model,X,y,cv=cv,scoring ="accuracy")
print(scores)
print("Average Accuracy ",round(scores.mean(),4))
print("Standard Daviation ",round(scores.std(),4))

[0.73175 0.72075 0.7305  0.7225  0.729  ]
Average Accuracy  0.7269
Standard Daviation  0.0044


In [40]:
param_grid = {
    "criterion":["gini","entropy"],
    "max_depth":[3,5,7,10,15,20, None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,6]
}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train,y_train)
print("Best parameter = ",grid.best_params_)
print("Best SV Score ",round(grid.best_score_,4))
grid_prediction = grid.predict(X_test)
print("Grid Search SV Test Accuracy ",round(accuracy_score(y_test,grid_prediction),4))

Best parameter =  {'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best SV Score  0.8262
Grid Search SV Test Accuracy  0.8427


In [42]:
# RandomizeSearchSV

param_dist = {
    "criterion":["gini","entropy"],
    "max_depth":randint(2,20),
    "min_samples_split":randint(2,15),
    "min_samples_leaf":randint(1,8)
}

random = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random.fit(X_train,y_train)

print("Best parameter = ",random.best_params_)
print("Best SV Score ",round(random.best_score_,4))

random_prediction = random.predict(X_test)

print("Random Search SV Test Accuracy ",round(accuracy_score(y_test,random_prediction),4))

Best parameter =  {'criterion': 'gini', 'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 4}
Best SV Score  0.8255
Random Search SV Test Accuracy  0.842
